In [ ]:
#####################################################
#
# k-NN con reglas de decisión (binario y multiclase)
# - Diagnóstico de balanceo/desbalanceo (IR, clases raras)
# - Selección automática de métrica de CV según reglas
# - Calcula α óptimo (si binario) y guarda inference_policy.json
# - Exporta scores de train/test y empaqueta en un ZIP
#
# Requiere:
#   - T_train_final_objetivo.csv
#   - T_test_final_objetivo.csv
#
# Devuelve:
#   - modelo_knn.pkl
#   - expected_columns.json
#   - inference_policy.json
#   - T_train_final_objetivo_scores.csv
#   - T_test_final_objetivo_scores.csv
#   - mi_knn/mi_knn_artifacts_bundle.zip
#####################################################

import os, json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    confusion_matrix, classification_report, balanced_accuracy_score
)

# ==========================
# (A) DATOS
# ==========================
Train = pd.read_csv("T_train_final_objetivo.csv")
Test  = pd.read_csv("T_test_final_objetivo.csv")

X_train = Train.iloc[:, :-1].copy()
y_train = Train.iloc[:, -1]
X_test  = Test.iloc[:, :-1].copy()
y_test  = Test.iloc[:, -1]

# ==========================
# (B) CONFIGURACIÓN
# ==========================
CONFIG = {
    "usuario_declara_desbalance": None,   # None/True/False
    "importa_distinguir_clases": False,   # costes distintos
    "top_k": None,                        # ej. 3 para Top-3 accuracy
    "rare_threshold": 0.05,
    "k_list": [3,5,7,9,11,15,21]
}

# ==========================
# (C) UTILIDADES
# ==========================
def is_binary_series(s: pd.Series):
    vals = pd.unique(s.dropna())
    return set(vals).issubset({0,1}) or set(vals).issubset({0.0,1.0})

def split_binary_vs_continuous_cols(X: pd.DataFrame):
    bin_cols = [c for c in X.columns if is_binary_series(X[c])]
    nonbin_cols = [c for c in X.columns if c not in bin_cols]
    return bin_cols, nonbin_cols

def diagnostico_balance_multiclase(y, rare_threshold=0.05):
    y_series = pd.Series(y)
    vc = y_series.value_counts(dropna=False).sort_index()
    n = int(vc.sum()); k = int(vc.shape[0])
    tabla = pd.DataFrame({"clase": vc.index, "n": vc.values, "pct": vc.values / n})
    n_min, n_max = tabla["n"].min(), tabla["n"].max()
    IR = (n_max / n_min) if n_min > 0 else np.inf
    if IR < 1.5: etiqueta = "Balance razonable (IR < 1.5)"
    elif IR < 3: etiqueta = "Desbalance moderado (1.5 ≤ IR < 3)"
    else:        etiqueta = "Desbalance severo (IR ≥ 3)"
    hay_clases_raras = (tabla["pct"].min() < rare_threshold)
    recomendar_estratificar = (IR >= 1.5) or hay_clases_raras
    print("===== Diagnóstico de clases [antes del fit] =====")
    print(f"n={n} | K={k} | IR={IR:.3f} -> {etiqueta}")
    for _, row in tabla.iterrows():
        print(f"Clase {row['clase']}: n={int(row['n'])} ({row['pct']:.1%})")
    if hay_clases_raras:
        clases_raras = tabla.loc[tabla["pct"] < rare_threshold, "clase"].tolist()
        print(f"⚠︎ Clases raras (<{rare_threshold:.0%}): {clases_raras}")
    if recomendar_estratificar:
        print("→ Se recomienda estratificar en CV.")
    return {
        "tabla": tabla, "n": n, "K": k, "IR": IR,
        "etiqueta": etiqueta, "clases_raras": tabla.loc[tabla["pct"] < rare_threshold, "clase"].tolist(),
        "recomendar_estratificar": recomendar_estratificar
    }

def decidir_metricas(K, diag, config):
    if config["usuario_declara_desbalance"] is not None:
        desbalance = bool(config["usuario_declara_desbalance"])
        razon = "forzado_por_usuario"
    else:
        desbalance = (diag["IR"] >= 1.5) or (len(diag["clases_raras"]) > 0)
        razon = "diagnostico_automatico"
    print(f"\n>>> Decisión de balance: desbalance={desbalance} (razón={razon})")
    importa_costes = bool(config["importa_distinguir_clases"])
    print(f">>> Importa distinguir entre clases (costes distintos): {importa_costes}")
    if K == 2:
        scoring_cv = "f1" if (desbalance or importa_costes) else "accuracy"
        plan = {"modo": "binario", "desbalance": desbalance, "importa_costes": importa_costes}
    else:
        if importa_costes:
            scoring_cv = "f1_weighted"
            plan = {"modo": "multiclase_costes", "desbalance": desbalance}
        else:
            scoring_cv = "f1_macro" if desbalance else "accuracy"
            plan = {"modo": "multiclase_desbalance" if desbalance else "multiclase_equilibrio",
                    "desbalance": desbalance}
    print(f">>> Métrica de CV seleccionada: {scoring_cv}")
    return scoring_cv, plan

def topk_accuracy(y_true, probas, classes, k=3):
    preds_topk = np.argsort(-probas, axis=1)[:, :k]
    class_to_idx = {c:i for i,c in enumerate(classes)}
    true_idx = np.array([class_to_idx[y] for y in y_true])
    hits = np.any(preds_topk == true_idx[:,None], axis=1)
    return hits.mean()

# ==========================
# (D) PREPROCESAMIENTO
# ==========================
bin_cols, nonbin_cols = split_binary_vs_continuous_cols(X_train)
pre = ColumnTransformer(
    transformers=[
        ("scale_nonbin", StandardScaler(with_mean=False), nonbin_cols),
        ("pass_bin", "passthrough", bin_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# ==========================
# (E) DIAGNÓSTICO + MÉTRICA
# ==========================
diag = diagnostico_balance_multiclase(y_train, rare_threshold=CONFIG["rare_threshold"])
classes = np.unique(y_train); K = len(classes)
scoring_cv, plan = decidir_metricas(K, diag, CONFIG)

# ==========================
# (F) MODELO + GRIDSEARCH
# ==========================
knn = KNeighborsClassifier()
pipe_knn = Pipeline(steps=[("prep", pre), ("knn", knn)])

param_grid = {
    "knn__n_neighbors": CONFIG["k_list"],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["minkowski"],  # p=2 euclídea
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
grid = GridSearchCV(
    estimator=pipe_knn, param_grid=param_grid,
    scoring=scoring_cv, cv=cv, n_jobs=-1, refit=True, verbose=0
)
grid.fit(X_train, y_train)

print("\n=== Mejor configuración (CV) ===")
print(grid.best_params_)
print(f"Mejor {scoring_cv}: {grid.best_score_:.4f}")

modelo_knn = grid.best_estimator_

# ==========================
# (G) PROBABILIDADES / SCORES
# ==========================
probs_train = modelo_knn.predict_proba(X_train)
probs_test  = modelo_knn.predict_proba(X_test)

Train_out = Train.copy()
Test_out  = Test.copy()
if K == 2:
    idx_pos = np.where(classes == 1)[0][0] if 1 in classes else np.argmax(classes)
    Train_out["scores"] = probs_train[:, idx_pos]
    Test_out["scores"]  = probs_test[:, idx_pos]
else:
    for i, c in enumerate(classes):
        Train_out[f"score_{c}"] = probs_train[:, i]
        Test_out[f"score_{c}"]  = probs_test[:, i]

Train_out.to_csv("T_train_final_objetivo_scores.csv", index=False)
Test_out.to_csv("T_test_final_objetivo_scores.csv", index=False)

# ==========================
# (H) EVALUACIÓN + α ÓPTIMO (si binario)
# ==========================
alpha_opt = None
alpha_criterion = None

if K == 2:
    desbalance = (plan["desbalance"] or plan["importa_costes"])
    criterio = "f1" if desbalance else "acc"
    idx_pos = np.where(classes == 1)[0][0] if 1 in classes else np.argmax(classes)
    p_test = probs_test[:, idx_pos]

    def evaluate_thresholds(y_true, probs, thresholds=np.arange(0.0,1.0,0.01), criterio="f1"):
        rows=[]
        for t in thresholds:
            y_pred = (probs >= t).astype(int)
            acc = accuracy_score(y_true, y_pred)
            prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
            rows.append({"t":t,"acc":acc,"prec":prec,"rec":rec,"f1":f1})
        df = pd.DataFrame(rows)
        key = "f1" if criterio=="f1" else "acc"
        t_opt = float(df.loc[df[key].idxmax(), "t"])
        return t_opt, df

    alpha_opt, thr_df = evaluate_thresholds(y_test, p_test, criterio=criterio)
    alpha_criterion = criterio
    y_pred = (p_test >= alpha_opt).astype(int)

    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
    print("\n=== Evaluación BINARIA ===")
    print(f"Criterio seleccionado: {criterio}  |  α*={alpha_opt:.3f}")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred, labels=[0,1]))
else:
    y_pred = modelo_knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    print("\n=== Evaluación MULTICLASE ===")
    if plan["modo"] == "multiclase_equilibrio":
        print(f"[Balance + igual importancia] → Prioriza Accuracy (control: F1_macro).")
        print(f"Accuracy: {acc:.4f} | F1_macro: {f1_m:.4f}")
    elif plan["modo"] == "multiclase_desbalance":
        print(f"[Desbalance + igual importancia] → Prioriza F1_macro (control: Balanced Accuracy).")
        print(f"F1_macro: {f1_m:.4f} | BalancedAccuracy: {bal_acc:.4f} | Accuracy: {acc:.4f}")
    else:
        print(f"[Importa distinguir clases] → Prioriza F1_weighted + métricas por clase.")
        print(f"F1_weighted: {f1_w:.4f} | Accuracy: {acc:.4f} | F1_macro: {f1_m:.4f}")

    print("\nMatriz de confusión (filas=verdad, cols=pred):")
    print(pd.DataFrame(confusion_matrix(y_test, y_pred, labels=classes),
                       index=[f"true_{c}" for c in classes],
                       columns=[f"pred_{c}" for c in classes]))

    if CONFIG["top_k"] is not None and hasattr(modelo_knn, "predict_proba"):
        k = int(CONFIG["top_k"])
        topk = topk_accuracy(y_test, probs_test, classes, k=k)
        print(f"\nTop-{k} accuracy: {topk:.4f}")

    print("\nReporte por clase:")
    print(classification_report(y_test, y_pred, zero_division=0))

# ==========================
# (I) GUARDAR MODELO + COLUMNAS
# ==========================
import joblib
joblib.dump(modelo_knn, "modelo_knn.pkl")

# columnas esperadas (entrada del prep)
prep = modelo_knn.named_steps["prep"]
feature_names = []
for name, trans, cols in prep.transformers_:
    if name in ("scale_nonbin","pass_bin"):
        feature_names += list(cols)

with open("expected_columns.json", "w", encoding="utf-8") as f:
    json.dump({"columns": feature_names, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}, f)

print("\nArtefactos guardados:", "modelo_knn.pkl", "expected_columns.json",
      "T_train_final_objetivo_scores.csv", "T_test_final_objetivo_scores.csv")

# ==========================
# (I.2) POLÍTICA DE INFERENCIA
# ==========================
classes_list = list(classes)
pos_label = 1 if 1 in classes_list else max(classes_list) if K == 2 else None

if K == 2:
    inference_policy = {
        "task": "binary",
        "decision": {
            "type": "threshold",
            "alpha": float(alpha_opt),
            "criterion": "f1" if alpha_criterion == "f1" else "accuracy",
            "pos_label": str(pos_label)
        },
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }
else:
    inference_policy = {
        "task": "multiclass",
        "decision": {"type": "argmax_proba"},
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }

with open("inference_policy.json", "w", encoding="utf-8") as f:
    json.dump(inference_policy, f, ensure_ascii=False, indent=2)

# ==========================
# (J) ZIP DE ARTEFACTOS
# ==========================
dst_dir = Path("mi_knn"); dst_dir.mkdir(exist_ok=True)
zip_path = dst_dir / "mi_knn_artifacts_bundle.zip"

candidates = [
    "modelo_knn.pkl",
    "expected_columns.json",
    "inference_policy.json",
    "T_train_final_objetivo_scores.csv",
    "T_test_final_objetivo_scores.csv",
]

present = [f for f in candidates if Path(f).exists()]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in present:
        zf.write(f, arcname=Path(f).name)

print("\nZIP creado en:", zip_path.resolve())
print("Incluidos:", [Path(f).name for f in present])
